# Sequence Analysis of Metalloproteinases

This notebook catalogs and analyzes sequences of metalloproteinases from different manufacturers and species.

## 1. Import Required Libraries

In [1]:
import os
import pandas as pd
from Bio import SeqIO, pairwise2
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio.Align import MultipleSeqAlignment
from Bio.Align.Applications import ClustalOmegaCommandline
import matplotlib.pyplot as plt
import numpy as np

# Install Biopython if not present
try:
    import Bio
except ImportError:
    !pip install biopython

c:\Users\ryang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\Bio\pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
c:\Users\ryang\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\Bio\Application\__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


## 2. Define Sequence Metadata and Accession IDs

In [2]:
# Define metadata for each sequence
metadata = {
    'P14780': {
        'name': 'MMP9',
        'species': 'Homo sapiens',
        'uniprot_link': 'https://www.uniprot.org/uniprotkb/P14780/entry',
        'manufacturers': {
            'Enzo': {
                'production': 'E. coli',
                'details': 'Active recombinant, catalytic/fibronectin domain, Phe107-Pro449, C-terminal purification tag',
                'residues': 'Phe107-Pro449'
            },
            'Sino Biological': {
                'production': 'HEK293 Cells',
                'details': 'Met1-Asp707',
                'residues': 'Met1-Asp707'
            }
        }
    },
    'P08253': {
        'name': 'MMP2',
        'species': 'Homo sapiens',
        'uniprot_link': 'https://www.uniprot.org/uniprotkb/P08253/entry',
        'manufacturers': {
            'Enzo': {
                'production': 'Yeast',
                'details': 'Active recombinant, catalytic/fibronectin domain, Tyr110-Asp452, C-terminal purification tag',
                'residues': 'Tyr110-Asp452'
            }
        }
    },
    'P33434': {
        'name': 'MMP2',
        'species': 'Mus musculus',
        'uniprot_link': 'https://www.uniprot.org/uniprotkb/P33434/entry',
        'manufacturers': {
            'Sino Biological': {
                'production': 'HEK293 Cells',
                'details': 'Met409-Cys662',
                'residues': 'Met409-Cys662'
            }
        }
    },
    'P78536': {
        'name': 'ADAM17',
        'species': 'Homo sapiens',
        'uniprot_link': 'https://www.uniprot.org/uniprotkb/P78536/entry',
        'manufacturers': {
            'Enzo': {
                'production': 'Insect cells',
                'details': 'Recombinant glycosylated catalytic domain, Pro18-Val477, C-terminal His-tag',
                'residues': 'Pro18-Val477'
            }
        }
    },
    'Q9Z1K9': {
        'name': 'ADAM17',
        'species': 'Rattus norvegicus',
        'uniprot_link': 'https://www.uniprot.org/uniprotkb/Q9Z1K9/entry',
        'manufacturers': {
            'Sino Biological': {
                'production': 'HEK293 Cells',
                'details': 'Met1-Asp563',
                'residues': 'Met1-Asp563'
            }
        }
    }
}

# Display metadata
pd.DataFrame.from_dict(metadata, orient='index')

,name,species,uniprot_link,manufacturers
P14780,MMP9,Homo sapiens,https://www.uniprot.org/uniprotkb/P14780/entry,"{'Enzo': {'production': 'E. coli', 'details': ..."
P08253,MMP2,Homo sapiens,https://www.uniprot.org/uniprotkb/P08253/entry,"{'Enzo': {'production': 'Yeast', 'details': 'A..."
P33434,MMP2,Mus musculus,https://www.uniprot.org/uniprotkb/P33434/entry,{'Sino Biological': {'production': 'HEK293 Cel...
P78536,ADAM17,Homo sapiens,https://www.uniprot.org/uniprotkb/P78536/entry,"{'Enzo': {'production': 'Insect cells', 'detai..."
Q9Z1K9,ADAM17,Rattus norvegicus,https://www.uniprot.org/uniprotkb/Q9Z1K9/entry,{'Sino Biological': {'production': 'HEK293 Cel...


## 3. Load Protein Sequences

In [3]:
# Define sequences
sequences = {
    'P14780': 'MSLWQPLVLVLLVLGCCFAAPRQRQSTLVLFPGDLRTNLTDRQLAEEYLYRYGYTRVAEMRGESKSLGPALLLLQKQLSLPETGELDSATLKAMRTPRCGVPDLGRFQTFEGDLKWHHHNITYWIQNYSEDLPRAVIDDAFARAFALWSAVTPLTFTRVYSRDADIVIQFGVAEHGDGYPFDGKDGLLAHAFPPGPGIQGDAHFDDDELWSLGKGVVVPTRFGNADGAACHFPFIFEGRSYSACTTDGRSDGLPWCSTTANYDTDDRFGFCPSERLYTQDGNADGKPCQFPFIFQGQSYSACTTDGRSDGYRWCATTANYDRDKLFGFCPTRADSTVMGGNSAGELCVFPFTFLGKEYSTCTSEGRGDGRLWCATTSNFDSDKKWGFCPDQGYSLFLVAAHEFGHALGLDHSSVPEALMYPMYRFTEGPPLHKDDVNGIRHLYGPRPEPEPRPPTTTTPQPTAPPTVCPTGPPTVHPSERPTAGPTGPPSAGPTGPPTAGPSTATTVPLSPVDDACNVNIFDAIAEIGNQLYLFKDGKYWRFSEGRGSRPQGPFLIADKWPALPRKLDSVFEERLSKKLFFFSGRQVWVYTGASVLGPRRLDKLGLGADVAQVTGALRSGRGKMLLFSGRRLWRFDVKAQMVDPRSASEVDRMFPGVPLDTHDVFQYREKAYFCQDRFYWRVSSRSELNQVDQVGYVTYDILQCPED',
    'P08253': 'MEALMARGALTGPLRALCLLGCLLSHAAAAPSPIIKFPGDVAPKTDKELAVQYLNTFYGCPKESCNLFVLKDTLKKMQKFFGLPQTGDLDQNTIETMRKPRCGNPDVANYNFFPRKPKWDKNQITYRIIGYTPDLDPETVDDAFARAFQVWSDVTPLRFSRIHDGEADIMINFGRWEHGDGYPFDGKDGLLAHAFAPGTGVGGDSHFDDDELWTLGEGQVVRVKYGNADGEYCKFPFLFNGKEYNSCTDTGRSDGFLWCSTTYNFEKDGKYGFCPHEALFTMGGNAEGQPCKFPFRFQGTSYDSCTTEGRTDGYRWCGTTEDYDRDKKYGFCPETAMSTVGGNSEGAPCVFPFTFLGNKYESCTSAGRSDGKMWCATTANYDDDRKWGFCPDQGYSLFLVAAHEFGHAMGLEHSQDPGALMAPIYTYTKNFRLSQDDIKGIQELYGASPDIDLGTGPTPTLGPVTPEICKQDIVFDGIAQIRGEIFFFKDRFIWRTVTPRDKPMGPLLVATFWPELPEKIDAVYEAPQEEKAVFFAGNEYWIYSASTLERGYPKPLTSLGLPPDVQRVDAAFNWSKNKKTYIFAGDKFWRYNEVKKKMDPGFPKLIADAWNAIPDNLDAVVDLQGGGHSYFFKGAYYLKLENQSLKSVKFGSIKSDWLGC',
    'P33434': 'MEARVAWGALAGPLRVLCVLCCLLGRAIAAPSPIIKFPGDVAPKTDKELAVQYLNTFYGCPKESCNLFVLKDTLKKMQKFFGLPQTGDLDQNTIETMRKPRCGNPDVANYNFFPRKPKWDKNQITYRIIGYTPDLDPETVDDAFARALKVWSDVTPLRFSRIHDGEADIMINFGRWEHGDGYPFDGKDGLLAHAFAPGTGVGGDSHFDDDELWTLGEGQVVRVKYGNADGEYCKFPFLFNGREYSSCTDTGRSDGFLWCSTTYNFEKDGKYGFCPHEALFTMGGNADGQPCKFPFRFQGTSYNSCTTEGRTDGYRWCGTTEDYDRDKKYGFCPETAMSTVGGNSEGAPCVFPFTFLGNKYESCTSAGRNDGKVWCATTTNYDDDRKWGFCPDQGYSLFLVAAHEFGHAMGLEHSQDPGALMAPIYTYTKNFRLSHDDIKGIQELYGPSPDADTDTGTGPTPTLGPVTPEICKQDIVFDGIAQIRGEIFFFKDRFIWRTVTPRDKPTGPLLVATFWPELPEKIDAVYEAPQEEKAVFFAGNEYWVYSASTLERGYPKPLTSLGLPPDVQQVDAAFNWSKNKKTYIFAGDKFWRYNEVKKKMDPGFPKLIADSWNAIPDNLDAVVDLQGGGHSYFFKGAYYLKLENQSLKSVKFGSIKSDWLGC',
    'P78536': 'MRQSLLFLTSVVPFVLAPRPPDDPGFGPHQRLEKLDSLLSDYDILSLSNIQQHSVRKRDLQTSTHVETLLTFSALKRHFKLYLTSSTERFSQNFKVVVVDGKNESEYTVKWQDFFTGHVVGEPDSRVLAHIRDDDVIIRINTDGAEYNIEPLWRFVNDTKDKRMLVYKSEDIKNVSRLQSPKVCGYLKVDNEELLPKGLVDREPPEELVHRVKRRADPDPMKNTCKLLVVADHRFYRYMGRGEESTTTNYLIELIDRVDDIYRNTSWDNAGFKGYGIQIEQIRILKSPQEVKPGEKHYNMAKSYPNEEKDAWDVKMLLEQFSFDIAEEASKVCLAHLFTYQDFDMGTLGLAYVGSPRANSHGGVCPKAYYSPVGKKNIYLNSGLTSTKNYGKTILTKEADLVTTHELGHNFGAEHDPDGLAECAPNEDQGGKYVMYPIAVSGDHENNKMFSNCSKQSIYKTIESKAQECFQERSNKVCGNSRVDEGEECDPGIMYLNNDTCCNSDCTLKEGVQCSDRNSPCCKNCQFETAQKKCQEAINATCKGVSYCTGNSSECPPPGNAEDDTVCLDLGKCKDGKCIPFCEREQQLESCACNETDNSCKVCCRDLSGRCVPYVDAEQKNLFLRKGKPCTVGFCDMNGKCEKRVQDVIERFWDFIDQLSINTFGKFLADNIVGSVLVFSLIFWIPFSILVHCVDKKLDKQYESLSLFHPSNVEMLSSMDSASVRIIKPFPAPQTPGRLQPAPVIPSAPAAPKLDHQRMDTIQEDPSTDSHMDEDGFEKDPFPNSSTAAKSFEDLTDHPVTRSEKAASFKLQRQNRVDSKETEC',
    'Q9Z1K9': 'MRQRLLFLTTLVPFVLAPRPPEEPGSGSHLRLEKLDSLLSDYDILSLSNIQQHSIRKRDLQSATHLETLLTFSALKRHFKLYLTSSTERFSQNLRVVVVDGKEESEYSVKWQDFFSGHVVGEPDSRVLAHIGDDDVTVRINTDGAEYNIEPLWRFVNDTKDKRMLVYKSEDIKDFSRLQSPKVCGYLNADSEELLPKGLIDREPSEEFVRRVKRRAEPNPLKNTCKLLVVADHRFYKYMGRGEESTTTNYLIELIDRVDDIYRNTSWDNAGFKGYGVQIEQIRILKSPQEVKPGERHFNMAKSFPNEEKDAWDVKMLLEQFSLDIAEEASKVCLAHLFTYQDFDMGTLGLAYVGSPRANSHGGVCPKAYYNPGVKKNIYLNSGLTSTKNYGKTILTKEADLVTTHELGHNFGAEHDPDGLAECAPNEDQGGKYVMYPIAVSGDHENNKMFSNCSKQSIYKTIESKAQECFQERSNKVCGNSRVDEGEECDPGIMYLNNDTCCNSDCTLKPGVQCSDRNSPCCKNCQFETAQKKCQEAINATCKGVSYCTGNSSECPPPGDAEDDTVCLDLGKCKAGKCIPFCKREQELESCACADTDNSCKVCCRNLSGPCVPYVDAEQKNLFLRKGKPCTVGFCDMNGKCEKRVQDVIERFWDFIDQLSINTFGKFLADNIVGSVLVFSLIFWIPFSILVHCVDKKLDKQYESLSLFHHSNIEMLSSMDSASVRIIKPFPAPQTPGRLQALQPAAMMPPVSAAPKLDHQRMDTIQEDPSTDSHVDDDGFEKDPFPNSSAAAKSFEDLTDHPVTRSEKAASFKLQRQSRVDSKETEC'
}

# Create SeqRecord objects
seq_records = {}
for acc, seq in sequences.items():
    seq_records[acc] = SeqRecord(Seq(seq), id=acc, name=metadata[acc]['name'], description=f"{metadata[acc]['species']} {metadata[acc]['name']}")

print("Sequences loaded:")
for acc, record in seq_records.items():
    print(f"{acc}: {len(record.seq)} amino acids")

Sequences loaded:
P14780: 707 amino acids
P08253: 660 amino acids
P33434: 662 amino acids
P78536: 824 amino acids
Q9Z1K9: 827 amino acids


## 4. Catalog Sequences and Compute Biochemical Features

In [4]:
# Compute features
features = []
for acc, record in seq_records.items():
    seq = str(record.seq)
    length = len(seq)
    mw = sum({'A':89, 'R':174, 'N':132, 'D':133, 'C':121, 'Q':146, 'E':147, 'G':75, 'H':155, 'I':131, 'L':131, 'K':146, 'M':149, 'F':165, 'P':115, 'S':105, 'T':119, 'W':204, 'Y':181, 'V':117}.get(aa, 0) for aa in seq)  # approximate MW
    features.append({
        'Accession': acc,
        'Protein': metadata[acc]['name'],
        'Species': metadata[acc]['species'],
        'Length': length,
        'Molecular_Weight': mw
    })

features_df = pd.DataFrame(features)
features_df

,Accession,Protein,Species,Length,Molecular_Weight
0,P14780,MMP9,Homo sapiens,707,91078
1,P08253,MMP2,Homo sapiens,660,85661
2,P33434,MMP2,Mus musculus,662,85917
3,P78536,ADAM17,Homo sapiens,824,107730
4,Q9Z1K9,ADAM17,Rattus norvegicus,827,107780


## 5. Align Sequences to Identify Conserved and Variable Regions

In [5]:
# Perform pairwise alignments for orthologs
alignments = {}

# Human vs Mouse MMP2
human_mmp2 = seq_records['P08253'].seq
mouse_mmp2 = seq_records['P33434'].seq
alignments['MMP2_Human_vs_Mouse'] = pairwise2.align.globalxx(human_mmp2, mouse_mmp2, one_alignment_only=True)[0]

# Human vs Rat ADAM17
human_adam17 = seq_records['P78536'].seq
rat_adam17 = seq_records['Q9Z1K9'].seq
alignments['ADAM17_Human_vs_Rat'] = pairwise2.align.globalxx(human_adam17, rat_adam17, one_alignment_only=True)[0]

# Display alignment scores
for name, aln in alignments.items():
    print(f"{name}: Score = {aln.score}, Length = {len(aln.seqA)}")

# To visualize differences, find positions where they differ
def find_differences(aln):
    diffs = []
    for i, (a, b) in enumerate(zip(aln.seqA, aln.seqB)):
        if a != b:
            diffs.append((i+1, a, b))  # 1-based position
    return diffs

print("\nDifferences in MMP2 Human vs Mouse:")
for pos, a, b in find_differences(alignments['MMP2_Human_vs_Mouse']):
    print(f"Pos {pos}: {a} -> {b}")

print("\nDifferences in ADAM17 Human vs Rat:")
for pos, a, b in find_differences(alignments['ADAM17_Human_vs_Rat']):
    print(f"Pos {pos}: {a} -> {b}")

MMP2_Human_vs_Mouse: Score = 633.0, Length = 689
ADAM17_Human_vs_Rat: Score = 766.0, Length = 885

Differences in MMP2 Human vs Mouse:
Pos 3: A -> -
Pos 4: L -> -
Pos 5: M -> -
Pos 8: - -> V
Pos 9: - -> A
Pos 10: - -> W
Pos 14: T -> -
Pos 15: - -> A
Pos 20: A -> -
Pos 21: - -> V
Pos 24: L -> -
Pos 25: - -> V
Pos 27: G -> -
Pos 29: - -> C
Pos 32: S -> -
Pos 33: H -> -
Pos 34: A -> -
Pos 35: - -> G
Pos 36: - -> R
Pos 38: - -> I
Pos 158: F -> -
Pos 159: Q -> -
Pos 160: - -> L
Pos 161: - -> K
Pos 254: K -> -
Pos 255: - -> R
Pos 258: N -> -
Pos 260: - -> S
Pos 301: E -> -
Pos 302: - -> D
Pos 318: D -> -
Pos 319: - -> N
Pos 385: S -> -
Pos 386: - -> N
Pos 390: M -> -
Pos 391: - -> V
Pos 397: A -> -
Pos 398: - -> T
Pos 454: Q -> -
Pos 455: - -> H
Pos 467: A -> -
Pos 468: - -> P
Pos 472: I -> -
Pos 473: - -> A
Pos 475: L -> -
Pos 476: - -> T
Pos 477: - -> D
Pos 478: - -> T
Pos 529: M -> -
Pos 530: - -> T
Pos 568: I -> -
Pos 569: - -> V
Pos 594: R -> -
Pos 595: - -> Q
Pos 637: A -> -
Pos 638: -

## 6. Compare Domain Architecture and Cleavage Sites

In [6]:
# Domain architecture based on manufacturer info
domains = []
for acc, meta in metadata.items():
    for manuf, info in meta['manufacturers'].items():
        domains.append({
            'Accession': acc,
            'Protein': meta['name'],
            'Species': meta['species'],
            'Manufacturer': manuf,
            'Residue_Range': info['residues'],
            'Domain_Type': 'Catalytic/Fibronectin' if 'catalytic' in info['details'].lower() else 'Full Length',
            'Production': info['production']
        })

domains_df = pd.DataFrame(domains)
domains_df

,Accession,Protein,Species,Manufacturer,Residue_Range,Domain_Type,Production
0,P14780,MMP9,Homo sapiens,Enzo,Phe107-Pro449,Catalytic/Fibronectin,E. coli
1,P14780,MMP9,Homo sapiens,Sino Biological,Met1-Asp707,Full Length,HEK293 Cells
2,P08253,MMP2,Homo sapiens,Enzo,Tyr110-Asp452,Catalytic/Fibronectin,Yeast
3,P33434,MMP2,Mus musculus,Sino Biological,Met409-Cys662,Full Length,HEK293 Cells
4,P78536,ADAM17,Homo sapiens,Enzo,Pro18-Val477,Catalytic/Fibronectin,Insect cells
5,Q9Z1K9,ADAM17,Rattus norvegicus,Sino Biological,Met1-Asp563,Full Length,HEK293 Cells


## 7. Tabulate Manufacturer and Expression System Metadata

In [7]:
# Manufacturer comparison
manuf_data = []
for acc, meta in metadata.items():
    for manuf, info in meta['manufacturers'].items():
        manuf_data.append({
            'Accession': acc,
            'Protein': meta['name'],
            'Species': meta['species'],
            'Manufacturer': manuf,
            'Expression_System': info['production'],
            'Residue_Range': info['residues'],
            'Details': info['details']
        })

manuf_df = pd.DataFrame(manuf_data)
manuf_df

,Accession,Protein,Species,Manufacturer,Expression_System,Residue_Range,Details
0,P14780,MMP9,Homo sapiens,Enzo,E. coli,Phe107-Pro449,"Active recombinant, catalytic/fibronectin doma..."
1,P14780,MMP9,Homo sapiens,Sino Biological,HEK293 Cells,Met1-Asp707,Met1-Asp707
2,P08253,MMP2,Homo sapiens,Enzo,Yeast,Tyr110-Asp452,"Active recombinant, catalytic/fibronectin doma..."
3,P33434,MMP2,Mus musculus,Sino Biological,HEK293 Cells,Met409-Cys662,Met409-Cys662
4,P78536,ADAM17,Homo sapiens,Enzo,Insect cells,Pro18-Val477,"Recombinant glycosylated catalytic domain, Pro..."
5,Q9Z1K9,ADAM17,Rattus norvegicus,Sino Biological,HEK293 Cells,Met1-Asp563,Met1-Asp563


## 8. Save Sequence Catalogs and Difference Reports

In [8]:
# Save dataframes to CSV
features_df.to_csv('Sequence_Features.csv', index=False)
domains_df.to_csv('Domain_Architecture.csv', index=False)
manuf_df.to_csv('Manufacturer_Comparison.csv', index=False)

# Save differences to text file
with open('Sequence_Differences.txt', 'w') as f:
    f.write("Differences in MMP2 Human vs Mouse:\n")
    for pos, a, b in find_differences(alignments['MMP2_Human_vs_Mouse']):
        f.write(f"Pos {pos}: {a} -> {b}\n")
    
    f.write("\nDifferences in ADAM17 Human vs Rat:\n")
    for pos, a, b in find_differences(alignments['ADAM17_Human_vs_Rat']):
        f.write(f"Pos {pos}: {a} -> {b}\n")

print("Files saved: Sequence_Features.csv, Domain_Architecture.csv, Manufacturer_Comparison.csv, Sequence_Differences.txt")

Files saved: Sequence_Features.csv, Domain_Architecture.csv, Manufacturer_Comparison.csv, Sequence_Differences.txt
